# SARScope — interactive prototype
Select the **SARScope Submission (FP32)** kernel and run all cells. Use historical complete 12-month windows. Existing monthly cache is reused. New requests may take several minutes. This notebook is a live input interface, not a new training run.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, '/home/jovyan/vault/SARScope_Work/SARScope_Submission')
from pipeline import run_case
VAULT = Path('/home/jovyan/vault/SARScope_Work')
from IPython.display import display, Image
import ipywidgets as widgets
import pandas as pd

In [ ]:
lat = widgets.FloatText(value=49.4875, description='Latitude')
lon = widgets.FloatText(value=8.4660, description='Longitude')
month = widgets.Text(value='2021-01', description='Start month')
button = widgets.Button(description='Run 12-month case', button_style='primary')
log = widgets.Output()

def run_clicked(_):
    button.disabled = True
    with log:
        log.clear_output()
        try:
            case = run_case(VAULT, lat.value, lon.value, month.value)
            display(Image(filename=str(case / 'monthly_shares.png')))
            display(pd.read_csv(case / 'changes.csv').query("comparison == 'first_to_last'"))
            frames = sorted(case.glob('map_*.png'))
            selector = widgets.SelectionSlider(options=[(p.stem[4:],str(p)) for p in frames], description='Month')
            view = widgets.Output()
            def show(change=None):
                with view:
                    view.clear_output(wait=True)
                    display(Image(filename=selector.value, width=512))
            selector.observe(show, names='value')
            display(selector,view); show()
            print('Offline report and maps:',case)
        except Exception as exc:
            print(type(exc).__name__+': '+str(exc))
        finally:
            button.disabled = False

button.on_click(run_clicked)
display(widgets.VBox([lat,lon,month,button,log]))